In [ ]:
import os
import copy
import glob
import warnings # hide the warnings
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd                          
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


from scipy import sparse
from scipy.stats import lognorm
from scipy.stats import genextreme
from scipy.interpolate import griddata


from matplotlib.colors import BoundaryNorm
from matplotlib.colors import ListedColormap
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt 
from cartopy.io import shapereader   
import cartopy.crs as ccrs

from climada.hazard import Hazard
from climada.hazard import Centroids
from climada.hazard import TCTracks
from climada.hazard import TropCyclone
from climada.util.plot import plot_from_gdf

warnings.filterwarnings("ignore")

In [ ]:
###############################################################################################################################################
# below we use WRF (WRF has no synthetic)
###############################################################################################################################################

In [ ]:
files_list = sorted(glob.glob("/work/u1625133/tc-risk/PGW4K.v230926/wrfout_d01_*_48.nc")) # a list of all historical file names

In [ ]:
years = [int(os.path.basename(wrf_file).split("_")[2][:4]) for wrf_file in files_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
###############################################################################################################################################
# below we use TC Track data (txt files)
###############################################################################################################################################

In [ ]:
def compute_category_ms(vmax_ms):  # saffir_simpson_category
    if vmax_ms < 17.49:   # 34 kt
        return -1
    if vmax_ms < 32.92:   # 64 kt
        return 0
    if vmax_ms < 42.70:   # 83 kt
        return 1
    if vmax_ms < 49.39:   # 96 kt
        return 2
    if vmax_ms < 58.13:   # 113 kt
        return 3
    if vmax_ms < 70.48:   # 137 kt
        return 4
    return 5

In [ ]:
def compute_category_kt(vmax_kt): # saffir_simpson_category
    if vmax_kt < 34:
        return -1
    if vmax_kt < 64:
        return 0
    if vmax_kt < 83:
        return 1
    if vmax_kt < 96:
        return 2
    if vmax_kt < 113:
        return 3
    if vmax_kt < 137:
        return 4
    return 5   

In [ ]:
def build_track_data(track_df, event_code, id_no, basin="WP"):
    
    time = pd.to_datetime(track_df[["year", "month", "day", "hour"]]).to_numpy(dtype="datetime64[ns]")

    ds_track = xr.Dataset(coords=dict(time=time,
                                      lat=("time", track_df["lat"].values),
                                      lon=("time", track_df["lon"].values),
                                     ),
                          
                          data_vars=dict(max_sustained_wind=("time", track_df["vmax_ms"].values * 1.94384),
                                         central_pressure=("time", track_df["pmin_hpa"].values),
                                         environmental_pressure=("time", track_df["penv_hpa"].values),
                                         radius_max_wind=("time", np.full(len(time), np.nan)), # see "def estimate_rmw(rmw, cen_pres):"
                                         basin=("time", np.array([basin] * len(time))),
                                         time_step=("time", np.ones(len(time), dtype=float)),
                                        ),

                          attrs=dict(name=event_code[6:], # e.g., "XXXXXXMEKKHALA"
                                     max_sustained_wind_unit="kt",
                                     central_pressure_unit="hPa",
                                     sid=track_df.at[0, "sid"],
                                     id_no=id_no,
                                     orig_event_flag=True,
                                     category=compute_category_ms((track_df["vmax_ms"].values).max()),
                                    ),
                          )
    return ds_track

In [ ]:
tracks_list = sorted(glob.glob("/work/u1625133/tc-risk/tc_track_data/TCtrack.slp.4C_*_48.txt"))

In [ ]:
years = [int(os.path.basename(track_file).split("_")[1][:4]) for track_file in tracks_list]
num_of_years = (max(years) - min(years) + 1)

In [ ]:
histogram_x, histogram_y = np.unique(np.array(years), return_counts=True)
plt.bar(histogram_x, histogram_y)
plt.ylabel("#")
plt.xlabel("Year")

In [ ]:
###############################################################################################################################################
# below we find the overlapping time range in WRF and TC Track data
###############################################################################################################################################

In [ ]:
assert len(files_list) == len(tracks_list)

overlap_time_intersect = []


for wrf_file, track_file in zip(files_list, tracks_list):

    # Number of WRF time steps
    with xr.open_dataset(wrf_file) as ref:
        wrf_times = pd.to_datetime(ref["Times"].values.astype("U"),
                                   format="%Y-%m-%d_%H:%M:%S"
                                  )

    # Number of track time steps
    track_ref = pd.read_csv(track_file,
                            sep=r"\s+",
                            header=None,
                            comment="#"
                           )




    track_times = pd.to_datetime({"year":  track_ref.iloc[:, 1],
                                  "month": track_ref.iloc[:, 2],
                                  "day":   track_ref.iloc[:, 3],
                                  "hour":  track_ref.iloc[:, 4]}
                                 )




    overlap_times = wrf_times.intersection(track_times)

    overlap_time_intersect.append(overlap_times)

In [ ]:
###############################################################################################################################################
# create the WRF hourly hazard object
###############################################################################################################################################

In [ ]:
n_ev = len(files_list)

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0


with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
intensity_all_hours = []
event_name = []
for wrf_file, overlap in zip(files_list, overlap_time_intersect):
    
    event_pos_name = wrf_file.split("_")[2]  # e.g. "202002MEKKHALA"
    
    with xr.open_dataset(wrf_file) as ref:
        # intensity_all_hours
        ref = ref.swap_dims({"Time": "XTIME"}) # use "XTIME" as the dimension coordinate instead of "Time"
        u10_slice = ref["U10"].sel(XTIME=overlap)
        v10_slice = ref["V10"].sel(XTIME=overlap)

        u10_masked  = u10_slice.values[:, mask.values]
        v10_masked  = v10_slice.values[:, mask.values]

        wind_speed_hourly = np.sqrt(u10_masked**2 + v10_masked**2)        
        intensity_all_hours.append(wind_speed_hourly[1:, :]) # skip HOUR000
       
        # event_name
        event_all_hours = len(overlap)
        for event_hour_i in range(1, event_all_hours): # skip HOUR000
            event_name.append(f"{event_pos_name}_HOUR_{event_hour_i:03d}")
            

intensity = sparse.csr_matrix(np.vstack(intensity_all_hours))
assert len(event_name) == intensity.shape[0]
del intensity_all_hours
###############################################################################################################################################
frequency = []

for overlap in overlap_time_intersect:

    f = (  (  n_ev / (max(years) - min(years) + 1)  ) / n_ev  ) / (len(overlap) - 1)

    frequency.extend( [f] * (len(overlap) - 1)  )

frequency = np.array(frequency)
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
n_hours = sum(len(overlap) - 1 for overlap in overlap_time_intersect)
event_id = np.arange(1, n_hours + 1, dtype=int) 
###############################################################################################################################################
# date = ...

In [ ]:
haz_hourly = TropCyclone(centroids=centroids,
                         intensity=intensity,
                         frequency=frequency,
                         event_id=event_id,
                         event_name=event_name,
                         units=units)

In [ ]:
haz_hourly.check()

In [ ]:
for name in haz_hourly.event_name:
    if "198801SUSAN" in name:
        ax = haz_hourly.plot_intensity(name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

In [ ]:
###############################################################################################################################################
# the all_tracks_haz hazard object (without generating probabilistic synthetic events)
###############################################################################################################################################

In [ ]:
all_track_data = []

id_no = 1
for track_file in tracks_list:
    track_df = pd.read_csv(track_file,
                           sep=r"\s+",
                           header=None,
                           names=["sid","year","month","day","hour","lon","lat","vmax_ms","pmin_hpa","penv_hpa"]) # pandas.Dataframe

    filename = track_file.split("/")[-1] # the last element in ["", "lfs", "home", ..., "TCtrack.slp.4C_202002MEKKHALA_48.txt"] 
    event_code = filename.split("_")[1] # the second element in ["TCtrack.slp.4C", "202002MEKKHALA", "48.txt"]

    track_data = build_track_data(track_df, event_code, id_no, basin="WP")
    all_track_data.append(track_data)


    id_no += 1

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

all_tracks = TCTracks(data=all_track_data)
all_tracks.plot().set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree()) # boundary the same as WRF

In [ ]:
# centroids = from WRF
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()])
###############################################################################################################################################
# other Hazard variables 
# ...

In [ ]:
all_tracks_haz = TropCyclone.from_tracks(tracks=all_tracks, 
                                         centroids=centroids, # centroids = the same as WRF
                                         model='H1980', 
                                         model_kwargs={"gradient_to_surface_winds": 0.9},
                                         intensity_thres=0,
                                         store_windfields=True) 

In [ ]:
all_tracks_haz.check()

In [ ]:
all_tracks_haz.plot_intensity(0, vmin=0, vmax=70)
all_tracks_haz.plot_intensity(1)
all_tracks_haz.plot_intensity(2)
all_tracks_haz.plot_intensity("202002MEKKHALA", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202003BAVI", vmin=0, vmax=40)
all_tracks_haz.plot_intensity("202004ATSANI", vmin=0, vmax=40)

In [ ]:
###############################################################################################################################################
# CNN  XXXX hours <-> XXXX hours
###############################################################################################################################################

In [ ]:
class CNN(nn.Module): 
    def __init__(self, X_mean_tensor, X_std_tensor, mask_rect_tensor):
        super().__init__() # nn.Module.__init__(self)

        self.net    = nn.Sequential(nn.Conv2d(3, 16, kernel_size=(3, 3), stride=1, padding=1),  # input: (n_hours, channels, nlat, nlon) -> (n_hours, 16, nlat, nlon)                               
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(16, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 16, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 32, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 32, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(32, 16, kernel_size=(3, 3), stride=1, padding=1), # (n_hours, 32, nlat, nlon) -> (n_hours, 16, nlat, nlon)
                                    nn.ReLU(), # activation function
                                    # no max pooling after ReLU


                                    nn.Conv2d(16, 1, kernel_size=(3, 3), stride=1, padding=1) # output: res_pred 
                                                                                              # output: (n_hours, 1, nlat, nlon)
                                                                                              # Y_pred = X + res_pred
                                    )

        self.register_buffer("X_mean_tensor", X_mean_tensor)
        self.register_buffer("X_std_tensor", X_std_tensor)
        self.register_buffer("mask_rect_tensor", mask_rect_tensor)


    def forward(self, X):
        
        normalized_X = (X - self.X_mean_tensor) / self.X_std_tensor # X is of torch.tensor
        
        normalized_X[:, :, ~self.mask_rect_tensor] = 0.0     
        
        return self.net(normalized_X) # meaning nn.Sequential(...)(normalized_X)

In [ ]:
###############################################################################################################################################
# CNN channel: "all_tracks_haz_hourly" 
###############################################################################################################################################

In [ ]:
assert n_ev == len(tracks_list)

In [ ]:
# centroids = from WRF 
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]
                         )
###############################################################################################################################################
n_hours = sum( len(overlap) - 1 for overlap in overlap_time_intersect )

n_centroids = centroids.size # from WRF

intensity_all_hours = np.empty((n_hours, n_centroids), dtype=float)

row_i = 0

for event_pos in range(len(overlap_time_intersect)):
    
    event_all_hours = len(overlap_time_intersect[event_pos])
    
    for event_hour_i in range(1, event_all_hours): # skip HOUR000
        wind_vec_hour = (all_tracks_haz.windfields[event_pos][event_hour_i, :]
                         .toarray()
                         .reshape(n_centroids, 2)
                        ) # u_wind and v_wind per centroid
        
        u_wind = wind_vec_hour[:, 0] # first wind-vector component at each centroid
        v_wind = wind_vec_hour[:, 1] # second wind-vector component at each centroid
        
        wind_speed_hour = np.sqrt(u_wind**2 + v_wind**2)

        intensity_all_hours[row_i, :] = wind_speed_hour

        row_i += 1 # add one after filling each row

assert row_i == n_hours
intensity = sparse.csr_matrix(intensity_all_hours)

del intensity_all_hours
del all_tracks_haz.windfields
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
event_name = []
for track_file, overlap in zip(tracks_list, overlap_time_intersect):
    
    event_pos_name = Path(track_file).name.split("_")[1]  # e.g. "202002MEKKHALA"
    event_all_hours = len(overlap)

    for event_hour_i in range(1, event_all_hours): # skip HOUR000
        event_name.append(
            f"{event_pos_name}_HOUR_{event_hour_i:03d}"   )

assert len(event_name) == n_hours    
###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
event_id = np.arange(1, n_hours + 1, dtype=int)  
###############################################################################################################################################
frequency = []
for overlap in overlap_time_intersect:

    f = (  (  n_ev / (max(years) - min(years) + 1)  ) / n_ev  ) / (len(overlap) - 1)

    frequency.extend( [f] * (len(overlap) - 1)  )

frequency = np.array(frequency)



In [ ]:
all_tracks_haz_hourly = TropCyclone(centroids=centroids,
                                    intensity=intensity,
                                    frequency=frequency,
                                    event_id=event_id,
                                    event_name=event_name,
                                    units=units)

In [ ]:
all_tracks_haz_hourly.check()

In [ ]:
for name in all_tracks_haz_hourly.event_name:
    if "198801SUSAN" in name:
        ax = all_tracks_haz_hourly.plot_intensity(name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

In [ ]:
###############################################################################################################################################
# CNN channel: era5 anomaly 
###############################################################################################################################################

In [ ]:
wind_anom_tw = xr.open_dataset("/work/u1625133/tc-risk/ERA5/wind_anom_tw.nc")

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]



pts_lat = xr.DataArray(lat, dims="points")
pts_lon = xr.DataArray(lon, dims="points") 




regrid = [] # regrid[0] has all hours in event 0, 
            # ...,  
            # regrid[196] has all hours in event 196
for overlap in overlap_time_intersect:

    track_times = pd.DatetimeIndex(overlap)[1:] # skip HOUR000

    wind_anom_tw_time_hourly = wind_anom_tw.sel(time=track_times)


    # sort before interpolation
    wind_anom_tw_time_hourly = wind_anom_tw_time_hourly.sortby(["latitude", "longitude"])

    # interpolate ERA5 anomaly field to WRF mask points
    r = wind_anom_tw_time_hourly.interp(latitude=pts_lat,
                                        longitude=pts_lon,
                                        method="linear")

    regrid.append(r)

In [ ]:
###############################################################################################################################################
# CNN channel: terrain 
###############################################################################################################################################

In [ ]:
terr = xr.open_dataset("/work/u1625133/tc-risk/terrain/wrf_pgw_twn_grid_coords.nc")

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]




# terr
ter_lat = terr["XLAT"].values.ravel()
ter_lon = terr["XLONG"].values.ravel()
ter_values = terr["TER"].values.ravel()


# interpolation
# r_terrain is a 1d numpy array
r_terrain = griddata(points=(ter_lon, ter_lat), # from terr points
                     values=ter_values,
                     xi=(lon, lat), # to WRF mask points
                     method="linear")

In [ ]:
################################################################################################################################################
# getting ready for training
###############################################################################################################################################

In [ ]:
assert n_hours == all_tracks_haz_hourly.intensity.shape[0] # use the input X

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of (499*549)

y_idx, x_idx = np.where(mask.values)
ny = y_idx.max() - y_idx.min() + 1
nx = x_idx.max() - x_idx.min() + 1
mask_rect = mask.isel( south_north = slice(y_idx.min(), y_idx.max() + 1),
                       west_east   = slice(x_idx.min(), x_idx.max() + 1) ) # mask_rect is of (254*235)


X_cnn = np.zeros((n_hours, 3, ny, nx)) # (n_hours, 3 channels, ny, nx)
Y_cnn = np.zeros((n_hours, 1, ny, nx)) # (n_hours, 1 channel, ny, nx)

# X_cnn first channel
for s in range(0, n_hours, 256):
    X_cnn[s:s+256, 0, mask_rect.values] = all_tracks_haz_hourly.intensity[s:s+256].toarray()

# X_cnn second channel
i = 0
for r in regrid:
    n = r["sqrt_wind_speed_anomaly"].values.shape[0]
    X_cnn[i:i+n, 1, mask_rect.values] =  r["sqrt_wind_speed_anomaly"].values 
    i += n

# X_cnn third channel 
X_cnn[:, 2, mask_rect.values] = r_terrain 

# Y_cnn
for s in range(0, n_hours, 256):
    Y_cnn[s:s+256, 0, mask_rect.values] = haz_hourly.intensity[s:s+256].toarray()

res_cnn = Y_cnn - X_cnn[:, 0:1, :, :] # 0:1 selects the channel in X_cnn

In [ ]:
X_tensor   = torch.tensor(X_cnn, dtype=torch.float32)
Y_tensor   = torch.tensor(Y_cnn, dtype=torch.float32)
res_tensor = torch.tensor(res_cnn, dtype=torch.float32)

In [ ]:
torch.manual_seed(42)  # fix training, validation, test samples
                       # fix initial CNN weights

hours_per_event  = torch.tensor([len(overlap) - 1 
                                 for overlap in overlap_time_intersect])

# shuffle EVENTS, not HOURS
event_perm       = torch.randperm(len(overlap_time_intersect))
cumulative_hours = torch.cumsum(hours_per_event[event_perm], dim=0)

cut_70 = torch.argmin( torch.abs(cumulative_hours - 0.70 * n_hours) ).item() + 1
cut_85 = torch.argmin( torch.abs(cumulative_hours - 0.85 * n_hours) ).item() + 1

train_idx      = event_perm[:cut_70]
validation_idx = event_perm[cut_70:cut_85]
test_idx       = event_perm[cut_85:]

hour_idx_by_event = torch.split( torch.arange(n_hours), hours_per_event.tolist() )

train_idx      = torch.cat([hour_idx_by_event[i] for i in train_idx])
validation_idx = torch.cat([hour_idx_by_event[i] for i in validation_idx])
test_idx       = torch.cat([hour_idx_by_event[i] for i in test_idx])

In [ ]:
X_tensor_train        = X_tensor[train_idx]
X_tensor_validation   = X_tensor[validation_idx]
X_tensor_test         = X_tensor[test_idx]

res_tensor_train      = res_tensor[train_idx]
res_tensor_validation = res_tensor[validation_idx]

Y_tensor_test         = Y_tensor[test_idx]

In [ ]:
all_tensor_train      = TensorDataset(X_tensor_train, res_tensor_train)
all_tensor_validation = TensorDataset(X_tensor_validation, res_tensor_validation)


train_loader      = DataLoader(all_tensor_train, batch_size=64, shuffle=True) # shuffle at every epoch
validation_loader = DataLoader(all_tensor_validation, batch_size=64, shuffle=False) 

In [ ]:
X_mean        = X_cnn[train_idx][:, 0:3, mask_rect.values].mean(axis=(0, 2)) # axis=0: n_ev
                                                                             # axis=2: n_mask_points
X_mean        = X_mean.reshape(1, 3, 1, 1) 
X_mean_tensor = torch.tensor(X_mean, dtype=torch.float32)

X_std         = X_cnn[train_idx][:, 0:3, mask_rect.values].std(axis=(0, 2)) 
X_std         = X_std.reshape(1, 3, 1, 1) 
X_std_tensor  = torch.tensor(X_std, dtype=torch.float32)

mask_rect_tensor = torch.tensor(mask_rect.values, dtype=torch.bool)

In [ ]:
del X_cnn
del Y_cnn
del res_cnn

In [ ]:
################################################################################################################################################
# training + validation
###############################################################################################################################################

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(X_mean_tensor, X_std_tensor, mask_rect_tensor).to(device)

loss_func = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 200



for epoch in range(n_epochs):
    
    model.train()

    train_loss_all_batches = 0.0
    train_samples_all_batches = 0

    
    for x, r in train_loader: # mini batches per epoch
        
        optimizer.zero_grad() # clear gradients from previous batch


        X_tensor_train_batch = x.to(device)
        r_pred_batch = model(X_tensor_train_batch) # model.forward(X_tensor[train_idx])

        r_true_batch = r.to(device)
        
        train_loss_per_batch = loss_func(r_pred_batch[:, :, model.mask_rect_tensor],  
                                         r_true_batch[:, :, model.mask_rect_tensor])

        # learning weights
        train_loss_per_batch.backward() # computes the new gradients
        optimizer.step() # update model weights


        train_loss_all_batches += train_loss_per_batch.item() * x.size(0)
        train_samples_all_batches += x.size(0)

    mean_train_loss_per_epoch = train_loss_all_batches / train_samples_all_batches
   
    if epoch % 10 == 0: 
       # validation
       model.eval()

       validation_loss_all_batches = 0.0
       validation_samples_all_batches = 0
        
       with torch.no_grad():
           for x, r in validation_loader: 
               X_tensor_validation_batch = x.to(device)
               r_validation_pred_batch = model(X_tensor_validation_batch)
               
               r_validation_true_batch = r.to(device)
               
               validation_loss_per_batch = loss_func(r_validation_pred_batch[:, :, model.mask_rect_tensor], 
                                                     r_validation_true_batch[:, :, model.mask_rect_tensor])
               
               validation_loss_all_batches += validation_loss_per_batch.item() * x.size(0)
               validation_samples_all_batches += x.size(0)
           
           mean_validation_loss_this_epoch =  validation_loss_all_batches / validation_samples_all_batches


        

       print(f"epoch {epoch}, "
             f"train mse: {mean_train_loss_per_epoch:.6f}, "
             f"validation mse: {mean_validation_loss_this_epoch:.6f}")

In [ ]:
################################################################################################################################################
# testing
###############################################################################################################################################

In [ ]:
model.eval()
with torch.no_grad():
   
    # model predicts residual
    X_tensor_test_gpu = X_tensor_test.to(device)
    r_pred_test_gpu = model(X_tensor_test_gpu)
    # final corrected wind
    Y_pred_test_gpu = X_tensor_test_gpu[:, 0:1, :, :] + r_pred_test_gpu


    Y_tensor_test_gpu = Y_tensor_test.to(device) # only 1 channel
    
    mae = torch.mean( torch.abs(Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                Y_pred_test_gpu[:, :, model.mask_rect_tensor]) )
    
    rmse = torch.sqrt( torch.mean((Y_tensor_test_gpu[:, :, model.mask_rect_tensor] -
                                   Y_pred_test_gpu[:, :, model.mask_rect_tensor]) ** 2) )

    old_mae = torch.mean( torch.abs(Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                    X_tensor_test_gpu[:, 0:1, model.mask_rect_tensor]) )
    
    old_rmse = torch.sqrt (torch.mean((Y_tensor_test_gpu[:, :, model.mask_rect_tensor] - 
                                       X_tensor_test_gpu[:, 0:1, model.mask_rect_tensor]) ** 2) )
    
    # after GPU calculations are finished:
    Y_pred_test = Y_pred_test_gpu.cpu()


In [ ]:
print("old MAE:", old_mae.item())
print("old RMSE:", old_rmse.item())
print("MAE:", mae.item())
print("RMSE:", rmse.item())

In [ ]:
del X_tensor_test_gpu
del r_pred_test_gpu
del Y_pred_test_gpu
del Y_tensor_test_gpu

In [ ]:
###############################################################################################################################################
# testing visualization
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of xarray.DataArray

    centroids = Centroids( lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                           lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()] )

###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
Y_pred_test_n_ev = len(Y_pred_test)  
###############################################################################################################################################
Y_pred_test_event_id = all_tracks_haz_hourly.event_id[test_idx.numpy()]   
###############################################################################################################################################
# from (n_events, 1, nlat, nlon) to (n_events, nlat, nlon)
Y_pred_test = Y_pred_test.squeeze(1).numpy() 

# from (n_events, nlat, nlon) to (n_events, n_centroids)
flat_Y_pred_test = Y_pred_test[:, mask_rect.values] 

# make negative predictions = 0
flat_Y_pred_test[flat_Y_pred_test < 0] = 0

# convert to intensity sparse matrix
Y_pred_test_intensity = sparse.csr_matrix(flat_Y_pred_test)
###############################################################################################################################################
Y_pred_test_frequency = all_tracks_haz_hourly.frequency[test_idx.numpy()]
###############################################################################################################################################
Y_pred_test_fraction = Y_pred_test_intensity.copy()
Y_pred_test_fraction.data.fill(1)
###############################################################################################################################################
Y_pred_test_event_name = [all_tracks_haz_hourly.event_name[i] for i in test_idx.numpy()] 
###############################################################################################################################################
# date = ...
###############################################################################################################################################
# orig = ...

In [ ]:
Y_pred_test_haz = Hazard(intensity=Y_pred_test_intensity,
                         fraction=Y_pred_test_fraction,
                         centroids=centroids,  
                         units=units,
                         frequency=Y_pred_test_frequency,  
                         event_id=Y_pred_test_event_id,  
                         event_name=Y_pred_test_event_name)

In [ ]:
Y_pred_test_haz.check()

In [ ]:
Y_pred_test_haz.event_name

In [ ]:
for name in Y_pred_test_haz.event_name:
    if "198801SUSAN" in name:
        ax = Y_pred_test_haz.plot_intensity(event=name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

In [ ]:
###############################################################################################################################################
# the "all_tracks_haz_hourly_synth" hazard object
###############################################################################################################################################

In [ ]:
# centroids = from WRF
with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)
           ) # mask is of xarray.DataArray

    centroids = Centroids(
                          lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                          lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]
                         )
###############################################################################################################################################
all_tracks.equal_timestep()  
nb_synth_tracks=10
all_tracks.calc_perturbed_trajectories(nb_synth_tracks=nb_synth_tracks) 
all_tracks_haz_synth = TropCyclone.from_tracks(tracks=all_tracks, 
                                               centroids=centroids, # centroids = from WRF
                                               model='H1980', 
                                               model_kwargs={"gradient_to_surface_winds": 0.9},
                                               intensity_thres=0,
                                               store_windfields=True)   
###############################################################################################################################################
n_hours = sum( len(overlap) - 1 for overlap in overlap_time_intersect ) * (nb_synth_tracks + 1)

n_centroids = centroids.size

intensity_all_hours = []

overlap_time_intersect_expand = [overlap
                                 for overlap in overlap_time_intersect
                                 for _ in range(nb_synth_tracks + 1)]

row_i = 0

for event_pos in range(len(overlap_time_intersect_expand)):
    
    event_all_hours = len(overlap_time_intersect_expand[event_pos])
    
    # all hours for this event, excluding HOUR000
    wind_vec_hour = (all_tracks_haz_synth.windfields[event_pos][1:event_all_hours, :]) 


    # [u0, v0, u1, v1, u2, v2, ...] per event, where 0, 1, 2... are centroids
    u_wind =  wind_vec_hour[:, 0::2]
    v_wind =  wind_vec_hour[:, 1::2]
    
    wind_speed_hour =  (u_wind.multiply(u_wind) + v_wind.multiply(v_wind)     )
    wind_speed_hour.data = np.sqrt(wind_speed_hour.data)

    intensity_all_hours.append(wind_speed_hour)


    row_i += event_all_hours - 1

assert row_i == n_hours
intensity = sparse.vstack(intensity_all_hours, format="csr")
del intensity_all_hours
del all_tracks_haz_synth.windfields
###############################################################################################################################################
fraction = intensity.copy()
fraction.data.fill(1)
###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
event_id = np.arange(1, n_hours + 1, dtype=int)  
###############################################################################################################################################
frequency = []

for overlap in overlap_time_intersect_expand:

    f = (  (n_ev / (max(years) - min(years)+1)) / ((nb_synth_tracks + 1) *n_ev)  
           / (len(overlap) - 1) 
        )

    frequency.extend( [f] * (len(overlap) - 1)  )

frequency = np.array(frequency)
###############################################################################################################################################
orig = []

for i in range(len(overlap_time_intersect_expand)):
    is_orig = (i % (nb_synth_tracks + 1) == 0)

    orig.extend(
        [is_orig] * (len(overlap_time_intersect_expand[i]) - 1)    )

orig = np.array(orig)
###############################################################################################################################################
event_name = []

for track_file, overlap in zip(tracks_list, overlap_time_intersect):

    base_name = Path(track_file).name.split("_")[1]
    event_all_hours = len(overlap)

    for i in range(nb_synth_tracks + 1):

        if i == 0:
            event_pos_name = base_name
        else:
            event_pos_name = f"{base_name}_gen_{i}"

        for event_hour_i in range(1, event_all_hours):  # skip HOUR000
            event_name.append(
                f"{event_pos_name}_HOUR_{event_hour_i:03d}"
            )

assert len(event_name) == n_hours

In [ ]:
all_tracks_haz_hourly_synth = Hazard(intensity=intensity,
                                     fraction=fraction,
                                     centroids=centroids,  
                                     units=units,
                                     frequency=frequency,  
                                     event_id=event_id,  
                                     event_name=event_name,
                                     orig=orig)

In [ ]:
all_tracks_haz_hourly_synth.check()

In [ ]:
###############################################################################################################################################
# apply the trained CNN to hourly synthetics
###############################################################################################################################################

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of (499*549)

y_idx, x_idx = np.where(mask.values)
ny = y_idx.max() - y_idx.min() + 1
nx = x_idx.max() - x_idx.min() + 1
mask_rect = mask.isel( south_north = slice(y_idx.min(), y_idx.max() + 1),
                       west_east   = slice(x_idx.min(), x_idx.max() + 1) ) # mask_rect is of (254*235)


exclude_historicals = ~all_tracks_haz_hourly_synth.orig
n_hours_exclude_historicals   = exclude_historicals.sum()
intensity_exclude_historicals = all_tracks_haz_hourly_synth.intensity[exclude_historicals]


X_synth = np.zeros((n_hours_exclude_historicals, 3, ny, nx)) 

# first channel 
for s in range(0, n_hours_exclude_historicals, 256): # 256 hours per batch 
    X_synth[s:s+256, 0, mask_rect.values] = intensity_exclude_historicals[s:s+256].toarray()
    
# second channel
i = 0
for r in regrid:
    n = r["sqrt_wind_speed_anomaly"].values.shape[0] 
    for _ in range(nb_synth_tracks):
        X_synth[i:i+n, 1, mask_rect.values] =  r["sqrt_wind_speed_anomaly"].values
        i += n
assert i == n_hours_exclude_historicals

# third channel
X_synth[:, 2, mask_rect.values] = r_terrain

X_synth_tensor = torch.tensor(X_synth, dtype=torch.float32) # (n_hours_exclude_historicals, 3 channels, ny, nx)

# for prediction Y
Y_synth_pred_tensor = torch.empty((n_hours_exclude_historicals, 1, ny, nx), dtype=torch.float32) 

In [ ]:
model.eval()
with torch.no_grad():
     for s in range(0, X_synth_tensor.shape[0], 256):  # 256 hours per batch

        # model predicts residual
        X_synth_tensor_batch_gpu = X_synth_tensor[s:s+256].to(device)
        r_batch_gpu = model(X_synth_tensor_batch_gpu)
    
        # final corrected wind
        Y_synth_pred_tensor_batch_gpu = X_synth_tensor_batch_gpu[:, 0:1, :, :] + r_batch_gpu
    
        # after GPU calculations are finished:
        Y_synth_pred_tensor[s:s+256]  = Y_synth_pred_tensor_batch_gpu.cpu()
        
        del X_synth_tensor_batch_gpu, r_batch_gpu, Y_synth_pred_tensor_batch_gpu

In [ ]:
###############################################################################################################################################
# taking max from Y_synth_pred_tensor
###############################################################################################################################################

In [ ]:
Y_synth_pred_max = torch.empty((n_ev * nb_synth_tracks, 1, Y_synth_pred_tensor.shape[2], Y_synth_pred_tensor.shape[3]),
                                dtype=Y_synth_pred_tensor.dtype   ) # (n_ev * nb_synth_tracks, 1 channel, ny, nx)

row_i = 0
event_pos = 0

for overlap in overlap_time_intersect:

    event_hours = len(overlap) - 1

    for _ in range(nb_synth_tracks):

        Y_synth_pred_max[event_pos] = (Y_synth_pred_tensor[row_i : row_i + event_hours]
                                       .max(dim=0)
                                       .values  )

        row_i += event_hours
        event_pos += 1


In [ ]:
###############################################################################################################################################
# the final "haz_synth" hazard object
###############################################################################################################################################

In [ ]:
historical_intensity_list = []

for f in files_list:
    with xr.open_dataset(f) as ref:
        u = ref["U10"].values[:, mask.values] 
        v = ref["V10"].values[:, mask.values] 
        
        max_wind = np.sqrt(u**2 + v**2).max(axis=0) # max on the time dimension
        historical_intensity_list.append(max_wind)

historical_intensity = sparse.csr_matrix(np.vstack(historical_intensity_list))

In [ ]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = ( (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
             (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max) ) # mask is of xarray.DataArray

    centroids = Centroids( lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()],
                           lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()] )

###############################################################################################################################################
units = 'm/s'
###############################################################################################################################################
haz_synth_n_ev = all_tracks_haz_synth.size
###############################################################################################################################################
haz_synth_event_id = all_tracks_haz_synth.event_id
###############################################################################################################################################
# from (n_events, 1, nlat, nlon) to (n_events, nlat, nlon)
Y_synth_pred_np = Y_synth_pred_max.squeeze(1).numpy() 

# from (n_events, nlat, nlon) to (n_events, n_centroids)
flat_Y_synth_pred = Y_synth_pred_np[:, mask_rect.values] 

# convert to intensity sparse matrix
haz_synth_intensity = sparse.csr_matrix(flat_Y_synth_pred)

# adding historicals back
blocks = []
j = 0
k = 0 
for i in all_tracks_haz_synth.orig:
    if i == True:
        blocks.append(historical_intensity[j])
        j += 1
    else:
        blocks.append(haz_synth_intensity[k])
        k += 1

haz_synth_intensity = sparse.vstack(blocks, format="csr")
###############################################################################################################################################
haz_synth_frequency = np.array([(n_ev / (max(years) - min(years) + 1)) / haz_synth_n_ev] * haz_synth_n_ev)
###############################################################################################################################################
haz_synth_fraction = haz_synth_intensity.copy()
haz_synth_fraction.data.fill(1)
###############################################################################################################################################
haz_synth_event_name = all_tracks_haz_synth.event_name

In [ ]:
haz_synth = Hazard(intensity=haz_synth_intensity,
                   fraction=haz_synth_fraction,
                   centroids=centroids,  
                   units=units,
                   frequency=haz_synth_frequency,  
                   event_id=haz_synth_event_id,  
                   event_name=haz_synth_event_name)

In [ ]:
haz_synth.check()

In [ ]:
haz_synth.write_hdf5("/work/u1625133/tc-risk/outputs/haz_synth.h5")

In [ ]:
for name in haz_synth.event_name:
    if "198801SUSAN" in name:
        ax = haz_synth.plot_intensity(event=name, vmin=0, vmax=50)
        cbar_ax = ax.get_figure().axes[-1]
        cbar_ax.set_yticks([0, 10, 20, 30, 40]) 

In [ ]:
###############################################################################################################################################
# exceedance_intensities
###############################################################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–10   pale yellow
                       "#fee391",  # 10–20  yellow
                       "#fec44f",  # 20–30  amber
                       "#fe9929",  # 30–35  orange
                       "#ec7014",  # 35–40  dark orange
                       "#cc4c02",  # 40–45  burnt orange
                       "#a63603",  # 45–50  brick
                       "#7f2704",  # 50–55  dark brick
                       "#67000d"]) # 55–60  deep red
                      
cmap.set_bad("lightgray")  

cmap.set_over("#3f0000") 

levels = [0, 10, 20, 30, 35, 40, 45, 50, 55, 60]
 
norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)

haz_synth.intensity = sparse.csc_matrix(haz_synth.intensity) 
exceedance_intensities, label, column_label = haz_synth.local_exceedance_intensity([10, 20, 30, 50]) # given return periods
haz_synth.intensity = sparse.csr_matrix(haz_synth.intensity) 


# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = exceedance_intensities.geometry.within(china)

value_cols = [col for col in exceedance_intensities.columns if col != "geometry"]

exceedance_intensities.loc[china_mask, value_cols] = np.nan

In [ ]:
###############################################################################################################################################
# plot exceedance_intensities 
###############################################################################################################################################

In [ ]:
axes = plot_from_gdf(exceedance_intensities, colorbar_name=label, title_subplots=column_label, cmap=cmap, norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
    fig.colorbar(pcolormesh, cax=cax, spacing="proportional", boundaries=levels, ticks=levels, label=label, extend="max")
    cax.tick_params(labelsize=6) 

plt.show()

In [ ]:
###############################################################################################################################################
# return periods
###############################################################################################################################################

In [ ]:
cmap = ListedColormap(["#fff7bc",  # 0–1   pale yellow
                       "#fee391",  # 1–2   yellow
                       "#fec44f",  # 2–3   amber
                       "#fe9929",  # 3–5   orange
                       "#ec7014",  # 5–8   dark orange
                       "#cc4c02",  # 8–12  burnt orange
                       "#a63603",  # 12–17 brick
                       "#7f2704",  # 17–23 dark brick
                       "#67000d"]) # 23–30 deep red
                      
cmap.set_bad("lightgray")

cmap.set_over("#3f0000") 

levels = [0, 1, 2, 3, 5, 8, 12, 17, 23, 30]

norm = BoundaryNorm(levels, ncolors=cmap.N, clip=False)


haz_synth.intensity = sparse.csc_matrix(haz_synth.intensity) 
return_periods, label, column_label = haz_synth.local_return_period([10, 15, 20, 25]) # given exceedance intensities
haz_synth.intensity = sparse.csr_matrix(haz_synth.intensity) 


# NaN caused by zero exceed_frequency -> infinite (1000000) return period
for threshold in [10, 15, 20, 25]:
    exceed = haz_synth.intensity > threshold

    exceed_frequency = np.asarray(
        exceed.T.dot(haz_synth.frequency)
    ).ravel()

    climada_nan = return_periods[str(threshold)].isna()
    
    return_periods.loc[climada_nan & (exceed_frequency == 0), str(threshold)] = 1000000


# mask People's Republic of China as nan
shapefile = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")

world = gpd.read_file(shapefile)

china = world[world.NAME == "China"].geometry.union_all()

china_mask = return_periods.geometry.within(china)

value_cols = [col for col in return_periods.columns if col != "geometry"]

return_periods.loc[china_mask, value_cols] = np.nan

In [ ]:
###############################################################################################################################################
# plot return periods
###############################################################################################################################################

In [ ]:
axes = plot_from_gdf(return_periods, colorbar_name=label, title_subplots=column_label, cmap=cmap, norm=norm)

fig = plt.gcf()

map_axes = list(axes.flat)

cbar_axes = [a for a in fig.axes if a not in map_axes]

for map_ax, cax in zip(map_axes, cbar_axes):
    pcolormesh = map_ax.collections[-1]   # the last collection is the data layer
    cax.clear()
    fig.colorbar(pcolormesh, cax=cax, spacing="proportional", boundaries=levels, ticks=levels, label=label, extend="max")
    cax.tick_params(labelsize=6) 

plt.show()